# Chapter 3.8: Attention Backends

## Learning Objectives

By the end of this notebook, you will:

1. Understand the AttentionBackend abstraction layer
2. Know when and why FlashAttention, FlashInfer, and xFormers backends are used
3. Understand the auto-selection logic for backends
4. Know what attention metadata each backend needs
5. Be able to compare performance across backends
6. Know how to add a new attention backend

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.8_attention_backends.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.8_attention_backends.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Attention Backend Abstraction

vLLM supports multiple attention implementations through a pluggable backend system. Different hardware and model configurations benefit from different implementations.

```
vllm/attention/
├── attention.py       # Unified Attention class (used by all models)
├── selector.py        # Auto-selects the best backend
├── backends/
│   ├── abstract.py    # AttentionBackend base class
│   ├── flash_attn.py  # FlashAttention v2
│   ├── flashinfer.py  # FlashInfer (paged attention optimized)
│   ├── xformers.py    # xFormers memory-efficient attention
│   ├── torch_sdpa.py  # PyTorch native scaled dot-product attention
│   └── rocm_flash_attn.py  # FlashAttention for AMD ROCm
└── ops/
    └── paged_attn.py   # PagedAttention CUDA kernel wrappers
```

In [ ]:
# Source: vllm/attention/backends/abstract.py

abstract_backend = '''
class AttentionBackend(ABC):
    """Abstract base class for attention backends."""
    
    @staticmethod
    @abstractmethod
    def get_name() -> str:
        """Return the name of this backend."""
        raise NotImplementedError
    
    @staticmethod
    @abstractmethod
    def get_impl_cls() -> Type["AttentionImpl"]:
        """Return the implementation class for this backend."""
        raise NotImplementedError
    
    @staticmethod
    @abstractmethod
    def get_metadata_cls() -> Type["AttentionMetadata"]:
        """Return the metadata class for this backend."""
        raise NotImplementedError


class AttentionImpl(ABC):
    """Abstract attention implementation."""
    
    @abstractmethod
    def forward(
        self,
        query: torch.Tensor,           # [num_tokens, num_heads, head_dim]
        key: torch.Tensor,             # [num_tokens, num_kv_heads, head_dim]
        value: torch.Tensor,           # [num_tokens, num_kv_heads, head_dim]
        kv_cache: torch.Tensor,        # KV cache tensor
        attn_metadata: AttentionMetadata,
        kv_scale: float = 1.0,
    ) -> torch.Tensor:
        """Compute attention.
        
        Must handle:
        1. PREFILL: Self-attention on new prompt tokens
           - Write new KV to cache
           - Compute causal attention
        2. DECODE: Attention over cached KV + new token
           - Write new KV to cache
           - PagedAttention: read from scattered blocks
        3. MIXED: Both prefill and decode in one batch
        """
        raise NotImplementedError
'''
print(abstract_backend)

## 2. The Unified Attention Class

In [ ]:
# Source: vllm/attention/attention.py

attention_class = '''
class Attention(nn.Module):
    """Unified attention layer used by ALL vLLM model implementations.
    
    This is the bridge between model code and attention backends.
    Models call this; it delegates to the selected backend.
    """
    
    def __init__(
        self,
        num_heads: int,
        head_size: int,
        scale: float,
        num_kv_heads: Optional[int] = None,
        alibi_slopes: Optional[List[float]] = None,
        cache_config: Optional[CacheConfig] = None,
        quant_config: Optional[QuantizationConfig] = None,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size
        self.scale = scale
        self.num_kv_heads = num_kv_heads or num_heads
        
        # Get the backend from the global selection
        self.backend = get_attn_backend()  # From selector.py
        
        # Create the backend-specific implementation
        impl_cls = self.backend.get_impl_cls()
        self.impl = impl_cls(
            num_heads=num_heads,
            head_size=head_size,
            scale=scale,
            num_kv_heads=self.num_kv_heads,
            alibi_slopes=alibi_slopes,
            ...
        )
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        kv_cache: Optional[torch.Tensor],
        attn_metadata: AttentionMetadata,
    ) -> torch.Tensor:
        """Forward pass: delegates to the selected backend."""
        return self.impl.forward(
            query, key, value, kv_cache, attn_metadata
        )
'''
print(attention_class)

## 3. Backend Selection Logic

In [ ]:
# Source: vllm/attention/selector.py

selector_code = '''
def get_attn_backend(
    num_heads: int,
    head_size: int,
    num_kv_heads: int,
    sliding_window: Optional[int],
    dtype: torch.dtype,
    kv_cache_dtype: Optional[str],
    block_size: int,
) -> Type[AttentionBackend]:
    """Select the best attention backend.
    
    Selection priority:
    1. User override via VLLM_ATTENTION_BACKEND env var
    2. Auto-selection based on capabilities
    """
    
    # Check for user override
    backend_name = os.environ.get("VLLM_ATTENTION_BACKEND", None)
    if backend_name:
        return _backend_name_to_cls(backend_name)
    
    # Auto-selection logic:
    
    # 1. Check for FlashInfer (if available and compatible)
    if _can_use_flashinfer(...):
        return FlashInferBackend
    
    # 2. Check for FlashAttention v2
    if _can_use_flash_attn(head_size, dtype, ...):
        return FlashAttentionBackend
    
    # 3. Fall back to xFormers
    if _can_use_xformers(...):
        return XFormersBackend
    
    # 4. Last resort: PyTorch SDPA
    return TorchSDPABackend


def _can_use_flash_attn(head_size, dtype, ...):
    """Check if FlashAttention can be used."""
    # Requirements:
    # 1. CUDA GPU with compute capability >= 8.0 (Ampere+)
    # 2. head_size in {64, 80, 96, 112, 128, 160, 192, 224, 256}
    # 3. dtype in {float16, bfloat16}
    # 4. flash_attn package installed
    if not is_ampere_or_newer():
        return False
    if head_size not in FLASH_ATTN_SUPPORTED_HEAD_SIZES:
        return False
    if dtype not in (torch.float16, torch.bfloat16):
        return False
    return True
'''
print(selector_code)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure: Backend Selection Flowchart (Decision Tree) ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
fig.patch.set_facecolor('white')

def diamond(ax, x, y, text, color=ORANGE):
    pts = [(x, y+0.6), (x+1.8, y), (x, y-0.6), (x-1.8, y)]
    poly = plt.Polygon(pts, facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(poly)
    ax.text(x, y, text, ha='center', va='center', fontsize=8, fontweight='bold', color='white')

def result_box(ax, x, y, text, color):
    rect = mpatches.FancyBboxPatch((x-1.4, y-0.4), 2.8, 0.8,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='white', linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

akw = dict(arrowstyle='->', color='#2C3E50', lw=1.8)

# Level 0: User override?
diamond(ax, 8, 9, 'VLLM_ATTENTION\n_BACKEND set?')
ax.annotate('Yes', xy=(12, 9), xytext=(9.8, 9), arrowprops=akw, fontsize=9, color=GREEN, fontweight='bold')
result_box(ax, 13.5, 9, 'Use specified\nbackend', '#95A5A6')

# Level 1: GPU arch
ax.annotate('No', xy=(8, 7.6), xytext=(8, 8.4), arrowprops=akw, fontsize=9, color=RED, fontweight='bold')
diamond(ax, 8, 7, 'GPU SM >= 8.0?\n(Ampere+)')

# No → xFormers or SDPA
ax.annotate('No', xy=(3, 7), xytext=(6.2, 7), arrowprops=akw, fontsize=9, color=RED, fontweight='bold')
diamond(ax, 3, 5.5, 'xFormers\ninstalled?')
ax.annotate('No', xy=(1, 5.5), xytext=(1.2, 5.5), arrowprops=akw, fontsize=9, color=RED, fontweight='bold')
result_box(ax, 1, 4.2, 'Torch SDPA\n(fallback)', '#95A5A6')
ax.annotate('Yes', xy=(3, 4.2), xytext=(3, 4.9), arrowprops=akw, fontsize=9, color=GREEN, fontweight='bold')
result_box(ax, 3, 3.5, 'xFormers', PURPLE)

# Yes → FlashInfer check
ax.annotate('Yes', xy=(8, 5.6), xytext=(8, 6.4), arrowprops=akw, fontsize=9, color=GREEN, fontweight='bold')
diamond(ax, 8, 5, 'FlashInfer\navailable?')

ax.annotate('Yes', xy=(12, 5), xytext=(9.8, 5), arrowprops=akw, fontsize=9, color=GREEN, fontweight='bold')
result_box(ax, 13.5, 5, 'FlashInfer\n(best decode)', GREEN)

# No → FlashAttention check
ax.annotate('No', xy=(8, 3.6), xytext=(8, 4.4), arrowprops=akw, fontsize=9, color=RED, fontweight='bold')
diamond(ax, 8, 3, 'head_size in\n{64..256}?\ndtype fp16/bf16?')

ax.annotate('Yes', xy=(12, 3), xytext=(9.8, 3), arrowprops=akw, fontsize=9, color=GREEN, fontweight='bold')
result_box(ax, 13.5, 3, 'FlashAttention v2\n(best prefill)', BLUE)

ax.annotate('No', xy=(8, 1.6), xytext=(8, 2.4), arrowprops=akw, fontsize=9, color=RED, fontweight='bold')
result_box(ax, 8, 1, 'xFormers\n(fallback)', PURPLE)

ax.set_title('Attention Backend Selection Decision Tree',
             fontsize=15, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# Backend comparison table

comparison = """
Attention Backend Comparison
════════════════════════════

┌────────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ Feature        │ FlashAttn v2 │ FlashInfer   │ xFormers     │ Torch SDPA   │
├────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
│ GPU Arch       │ Ampere+ (8.0)│ Ampere+ (8.0)│ Any CUDA     │ Any          │
│ Head sizes     │ 64-256       │ Flexible     │ Any          │ Any          │
│ Paged KV cache │ Yes (v2.5+)  │ Yes (native) │ Via custom   │ Limited      │
│ Prefill speed  │ Fastest      │ Fast         │ Good         │ Baseline     │
│ Decode speed   │ Good         │ Fastest      │ Good         │ Baseline     │
│ GQA/MQA        │ Yes          │ Yes          │ Yes          │ Yes          │
│ FP8 KV cache   │ Yes          │ Yes          │ No           │ No           │
│ Sliding window │ Yes          │ Yes          │ Yes          │ Limited      │
│ ALiBi          │ Yes          │ Yes          │ Yes          │ No           │
│ Chunked prefill│ Yes          │ Yes          │ Yes          │ Limited      │
└────────────────┴──────────────┴──────────────┴──────────────┴──────────────┘

Recommendation:
  - A100/H100 GPU: FlashAttention v2 or FlashInfer
  - Older GPU (V100): xFormers
  - CPU/Testing: Torch SDPA
  - High-throughput decode: FlashInfer (optimized paged attention)
  - Long-context prefill: FlashAttention v2 (best prefill perf)
"""
print(comparison)

## 4. FlashAttention Backend

In [ ]:
flash_attn_code = '''
# Source: vllm/attention/backends/flash_attn.py (simplified)

class FlashAttentionImpl(AttentionImpl):
    """FlashAttention v2 backend.
    
    Uses Tri Dao's flash-attn library for:
    - Prefill: flash_attn_varlen_func (IO-aware, fused softmax)
    - Decode: PagedAttention CUDA kernel (vLLM's custom kernel)
    """
    
    def forward(
        self,
        query, key, value, kv_cache, attn_metadata,
    ):
        # Step 1: Write new K,V to cache
        if kv_cache is not None:
            cache_ops.reshape_and_cache_flash(
                key, value, kv_cache,
                attn_metadata.slot_mapping,
            )
        
        # Step 2: Compute attention
        if attn_metadata.num_prefill_tokens > 0:
            # PREFILL: Use FlashAttention for self-attention
            # This is the fastest for prefill (IO-aware algorithm)
            prefill_output = flash_attn_varlen_func(
                q=query[:num_prefill],
                k=key[:num_prefill],
                v=value[:num_prefill],
                cu_seqlens_q=attn_metadata.query_start_loc,
                cu_seqlens_k=attn_metadata.seq_start_loc,
                max_seqlen_q=attn_metadata.max_prefill_seq_len,
                max_seqlen_k=attn_metadata.max_prefill_seq_len,
                softmax_scale=self.scale,
                causal=True,
            )
        
        if attn_metadata.num_decode_tokens > 0:
            # DECODE: Use PagedAttention kernel
            # This reads from scattered KV cache blocks
            decode_output = PagedAttention.forward_decode(
                query=query[num_prefill:],
                key_cache=kv_cache[0],
                value_cache=kv_cache[1],
                block_tables=attn_metadata.block_tables,
                seq_lens=attn_metadata.seq_lens_tensor,
                scale=self.scale,
            )
        
        # Step 3: Combine prefill and decode outputs
        output = torch.cat([prefill_output, decode_output])
        return output
'''
print(flash_attn_code)

## 5. FlashInfer Backend

In [ ]:
flashinfer_info = """
FlashInfer Backend
══════════════════

FlashInfer is a library optimized specifically for LLM serving.
Its key advantage: native paged attention that is faster than
vLLM's custom PagedAttention kernel for decode.

Key Features:
  1. Native paged KV cache support
     - Built from the ground up for non-contiguous memory
     - No separate PagedAttention kernel needed
  
  2. Batch-decode optimization
     - Specially optimized for the decode phase
     - Better occupancy for small batch decode
  
  3. Plan-execute pattern
     - "Plan" phase: pre-compute memory access patterns (CPU)
     - "Execute" phase: run the attention (GPU)
     - Plan can be reused if batch structure doesn't change

Usage:
  # Environment variable
  VLLM_ATTENTION_BACKEND=FLASHINFER vllm serve ...
  
  # Or in Python
  os.environ['VLLM_ATTENTION_BACKEND'] = 'FLASHINFER'

When to use FlashInfer:
  - High-throughput serving (many concurrent requests)
  - When decode latency is critical
  - Large batch sizes in decode phase
  - When using FP8 KV cache quantization
"""
print(flashinfer_info)

## 6. Performance Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated benchmark data (representative of real measurements)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Prefill throughput (tokens/s) by sequence length
seq_lens = [128, 512, 1024, 2048, 4096]
flash_attn_prefill = [150000, 120000, 90000, 60000, 35000]
flashinfer_prefill = [140000, 115000, 85000, 55000, 32000]
xformers_prefill = [100000, 80000, 60000, 40000, 22000]

ax1.plot(seq_lens, flash_attn_prefill, 'o-', label='FlashAttn v2', linewidth=2, color='#FF6B6B')
ax1.plot(seq_lens, flashinfer_prefill, 's-', label='FlashInfer', linewidth=2, color='#4ECDC4')
ax1.plot(seq_lens, xformers_prefill, '^-', label='xFormers', linewidth=2, color='#45B7D1')
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Prefill Throughput (tokens/s)')
ax1.set_title('Prefill Performance', fontweight='bold')
ax1.legend()
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)

# Decode throughput (tokens/s) by batch size
batch_sizes = [1, 4, 16, 64, 128, 256]
flash_attn_decode = [50, 180, 600, 2000, 3500, 5500]
flashinfer_decode = [55, 200, 700, 2400, 4200, 6800]
xformers_decode = [40, 150, 500, 1600, 2800, 4200]

ax2.plot(batch_sizes, flash_attn_decode, 'o-', label='FlashAttn v2', linewidth=2, color='#FF6B6B')
ax2.plot(batch_sizes, flashinfer_decode, 's-', label='FlashInfer', linewidth=2, color='#4ECDC4')
ax2.plot(batch_sizes, xformers_decode, '^-', label='xFormers', linewidth=2, color='#45B7D1')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Decode Throughput (tokens/s)')
ax2.set_title('Decode Performance', fontweight='bold')
ax2.legend()
ax2.set_xscale('log', base=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - FlashAttention v2 wins on prefill (IO-aware algorithm)")
print("  - FlashInfer wins on decode (optimized paged attention)")
print("  - xFormers is a solid fallback for older GPUs")
print("  - The gap widens with larger sequences/batches")

## 7. Adding a New Attention Backend

In [ ]:
new_backend_guide = '''
Adding a New Attention Backend
══════════════════════════════

Step 1: Create the backend file
  vllm/attention/backends/my_backend.py

Step 2: Define the metadata class
  class MyBackendMetadata(AttentionMetadata):
      # Add any backend-specific fields
      my_special_tensor: torch.Tensor

Step 3: Implement the backend
  class MyBackend(AttentionBackend):
      @staticmethod
      def get_name() -> str:
          return "MY_BACKEND"
      
      @staticmethod
      def get_impl_cls():
          return MyBackendImpl
      
      @staticmethod
      def get_metadata_cls():
          return MyBackendMetadata

Step 4: Implement the attention computation
  class MyBackendImpl(AttentionImpl):
      def forward(self, query, key, value, kv_cache, attn_metadata):
          # Handle prefill
          # Handle decode (with paged KV cache)
          # Handle mixed batches
          return output

Step 5: Register in selector.py
  Add to the selection logic in get_attn_backend()

Step 6: Test
  VLLM_ATTENTION_BACKEND=MY_BACKEND python -c "..."
'''
print(new_backend_guide)

## 8. How Prefill and Decode Differ

In [ ]:
prefill_vs_decode = """
Prefill vs Decode: Different Attention Patterns
════════════════════════════════════════════════

PREFILL (Prompt Processing):
  - Process all prompt tokens at once
  - Standard causal self-attention
  - Q, K, V are ALL from the current batch
  - Compute-bound (matrix multiplication)
  - Best with: FlashAttention (IO-aware, tiling)
  
  Q: [seq_len, num_heads, head_dim]  (all prompt tokens)
  K: [seq_len, num_kv_heads, head_dim]  (all prompt tokens)
  V: [seq_len, num_kv_heads, head_dim]  (all prompt tokens)
  → Standard flash_attn_varlen_func()

DECODE (Token Generation):
  - Process 1 new token per sequence
  - Q is the new token; K,V are from the CACHE
  - K,V are scattered across physical blocks
  - Memory-bound (loading KV from scattered blocks)
  - Best with: PagedAttention kernel or FlashInfer
  
  Q: [batch_size, num_heads, head_dim]  (1 per sequence)
  K_cache: scattered across physical blocks
  V_cache: scattered across physical blocks
  → PagedAttention.forward_decode()

MIXED BATCH (Chunked Prefill):
  - Some sequences are in prefill, some in decode
  - Need to handle both patterns in one forward pass
  - Backend must separate and handle each differently
"""
print(prefill_vs_decode)

## Exercises

### Exercise 1: Backend Selection Analysis

In [ ]:
# Exercise 1: Determine which backend is selected for various configurations

def select_backend(gpu_arch, head_size, dtype, has_flash_attn, has_flashinfer, has_xformers):
    """Simulate backend selection logic."""
    
    supported_flash_heads = {64, 80, 96, 112, 128, 160, 192, 224, 256}
    
    if has_flashinfer and gpu_arch >= 8.0 and dtype in ('float16', 'bfloat16'):
        return 'FlashInfer'
    
    if (has_flash_attn and gpu_arch >= 8.0 
        and head_size in supported_flash_heads 
        and dtype in ('float16', 'bfloat16')):
        return 'FlashAttention v2'
    
    if has_xformers:
        return 'xFormers'
    
    return 'Torch SDPA'

scenarios = [
    {'gpu_arch': 8.0, 'head_size': 128, 'dtype': 'float16', 
     'has_flash_attn': True, 'has_flashinfer': True, 'has_xformers': True},
    {'gpu_arch': 8.0, 'head_size': 128, 'dtype': 'float16',
     'has_flash_attn': True, 'has_flashinfer': False, 'has_xformers': True},
    {'gpu_arch': 7.0, 'head_size': 128, 'dtype': 'float16',
     'has_flash_attn': True, 'has_flashinfer': False, 'has_xformers': True},
    {'gpu_arch': 8.0, 'head_size': 48, 'dtype': 'float16',
     'has_flash_attn': True, 'has_flashinfer': False, 'has_xformers': True},
    {'gpu_arch': 7.0, 'head_size': 64, 'dtype': 'float32',
     'has_flash_attn': False, 'has_flashinfer': False, 'has_xformers': False},
]

gpu_names = {7.0: 'V100', 8.0: 'A100/H100'}

print(f"{'GPU':<12} {'Head':<6} {'DType':<10} {'FlashAttn':<10} {'FlashInfer':<12} {'Selected':<20}")
print("=" * 70)
for s in scenarios:
    backend = select_backend(**s)
    gpu = gpu_names.get(s['gpu_arch'], f"SM{s['gpu_arch']}")
    print(f"{gpu:<12} {s['head_size']:<6} {s['dtype']:<10} "
          f"{str(s['has_flash_attn']):<10} {str(s['has_flashinfer']):<12} {backend:<20}")

## Summary

Key takeaways:

1. **Pluggable backends**: vLLM abstracts attention behind AttentionBackend/AttentionImpl interfaces
2. **Auto-selection**: selector.py picks the best backend based on GPU, head size, dtype, and available libraries
3. **FlashAttention v2**: Best for prefill (IO-aware tiling algorithm)
4. **FlashInfer**: Best for decode (native paged attention optimization)
5. **xFormers**: Solid fallback for older GPUs or unsupported configurations
6. **Prefill vs Decode**: Different computational patterns require different optimizations
7. **Mixed batches**: Backends must handle both prefill and decode tokens in one pass

**Next**: Chapter 3.9 explores the Sampling and Logits Processing Pipeline.